# Practice P6: answers

Build each step yourself first. These are one way to write them; yours can look different and still be right, as long as the checks pass.

In [ ]:
import sys
sys.path.append("..")  # checks.py is one folder up
from checks import check

---

# Project 1: the shop till

## Project 1, step 1: read one line

Two cuts: `@` separates the price, then the first `x` separates quantity from name. The price is the fiddly part: rupees and paise are two separate whole numbers, and a single paise digit means tens of paise.

In [ ]:
def to_paise(price):
    price = price.strip()
    if "." not in price:
        return int(price) * 100
    rupees, paise = price.split(".")
    if len(paise) == 1:
        paise = paise + "0"          # "15.5" means 15 rupees 50 paise
    return int(rupees) * 100 + int(paise)


def parse_line(line):
    left, price = line.split("@")
    x_at = left.find("x")
    quantity = int(left[:x_at].strip())
    name = left[x_at + 1:].strip().lower()
    return (name, quantity, to_paise(price))

In [ ]:
check(parse_line("2 x tea @ 15.50"), ("tea", 2, 1550))
check(parse_line("1 x Masala Chai @ 20"), ("masala chai", 1, 2000))
check(parse_line("3 x wax candle @ 0.5"), ("wax candle", 3, 50))
check(parse_line("  10 x   samosa  @  12.05 "), ("samosa", 10, 1205))
check(parse_line("1 x water @ 0.05"), ("water", 1, 5))
check(parse_line("12 x xmas cookie @ 7"), ("xmas cookie", 12, 700))

## Project 1, step 2: show money

`//` and `%` by 100 split it into rupees and paise. The only trap is a single paise digit: 5 paise must show as `05`, not `5`.

In [ ]:
def money(paise):
    rupees = paise // 100
    rest = str(paise % 100)
    if len(rest) == 1:
        rest = "0" + rest
    return str(rupees) + "." + rest

In [ ]:
check(money(1550), "15.50")
check(money(3000), "30.00")
check(money(5), "0.05")
check(money(0), "0.00")
check(money(99), "0.99")
check(money(123456), "1234.56")
check(money(1009), "10.09")

## Project 1, step 3: the bill, with discounts

Combine first, then price. A dictionary from name to `[quantity, price]` merges repeated items. Then each item gets its own 10% check, and the big-bill rule looks at the whole total once at the end. The helper `combine` is reused by the receipt in step 4.

In [ ]:
def combine(lines):
    # name -> [quantity, price], in the order items first appear
    items = {}
    for line in lines:
        if line.strip() == "":
            continue
        name, quantity, price = parse_line(line)
        if name in items:
            items[name][0] += quantity
        else:
            items[name] = [quantity, price]
    return items


def item_discount(quantity, item_total):
    if quantity >= 3:
        return item_total // 10
    return 0


def bill_total(lines):
    total = 0
    for name, (quantity, price) in combine(lines).items():
        item_total = quantity * price
        total += item_total - item_discount(quantity, item_total)
    if total >= 50000:
        total -= 5000
    return total

In [ ]:
check(bill_total(["2 x tea @ 15.50", "1 x cake @ 120"]), 15100)
check(bill_total(["2 x tea @ 15.50", "1 x tea @ 15.50"]), 4185)
check(bill_total(["10 x coffee @ 60"]), 49000)
check(bill_total(["2 x box @ 250"]), 45000)          # exactly 500: big-bill discount applies
check(bill_total(["5 x box @ 100"]), 45000)          # 10% off brings it under 500: no big-bill discount
check(bill_total(["3 x pen @ 3.33"]), 900)           # 999 minus 99 (10% rounded down)
check(bill_total(["", "   ", "1 x milk @ 30"]), 3000)
check(bill_total([]), 0)

## Project 1, step 4: ⭐ Stretch: print the receipt

Build a list of 24-character rows with one helper, `row`, and join them at the end. The total discount doesn't need its own rules: it's "price before discounts" minus `bill_total`, so the big-bill ₹50 is included automatically.

In [ ]:
WIDTH = 24


def row(left, right):
    spaces = WIDTH - len(left) - len(right)
    return left + " " * spaces + right


def receipt(lines):
    rows = []
    full_price = 0
    for name, (quantity, price) in combine(lines).items():
        item_total = quantity * price
        full_price += item_total
        rows.append(row(name + " x" + str(quantity), money(item_total)))
    rows.append("-" * WIDTH)

    total = bill_total(lines)
    discount = full_price - total
    if discount > 0:
        rows.append(row("Discount", "-" + money(discount)))
    rows.append(row("TOTAL", money(total)))
    return "\n".join(rows)


print(receipt(["2 x tea @ 15.50", "1 x cake @ 120", "1 x tea @ 15.50"]))

In [ ]:
check(receipt(["2 x tea @ 15.50", "1 x cake @ 120", "1 x tea @ 15.50"]),
      "tea x3             46.50\n"
      "cake x1           120.00\n"
      "------------------------\n"
      "Discount           -4.65\n"
      "TOTAL             161.85")

check(receipt(["1 x milk @ 30"]),
      "milk x1            30.00\n"
      "------------------------\n"
      "TOTAL              30.00")

check(receipt(["10 x coffee @ 60"]),
      "coffee x10        600.00\n"
      "------------------------\n"
      "Discount         -110.00\n"
      "TOTAL             490.00")

check(receipt([]),
      "------------------------\n"
      "TOTAL               0.00")

---

# Project 2: the text analyser

## Project 2, step 1: split text into clean words

Turn every separator into a space, then let `.split()` do the cutting: with no argument it handles runs of spaces and never gives empty words.

In [ ]:
def words(text):
    cleaned = ""
    for ch in text.lower():
        if ch.isalnum() or ch == "'":
            cleaned += ch
        else:
            cleaned += " "
    return cleaned.split()

In [ ]:
check(words("Don't stop... believing!"), ["don't", "stop", "believing"])
check(words("well-known e-mail"), ["well", "known", "e", "mail"])
check(words("3 cats & 2 dogs"), ["3", "cats", "2", "dogs"])
check(words("Hello, hello HELLO"), ["hello", "hello", "hello"])
check(words("one\ntwo\tthree"), ["one", "two", "three"])
check(words(""), [])
check(words("  ?!  "), [])

## Project 2, step 2: the most used words

Count, then sort by two things at once. A sort key of `(-count, word)` puts high counts first and breaks ties alphabetically. `most_common` isn't enough here, because it doesn't sort ties by name.

In [ ]:
from collections import Counter


def top_words(text, n, ignore):
    counts = Counter()
    for word in words(text):
        if word not in ignore:
            counts[word] += 1
    pairs = sorted(counts.items(), key=lambda pair: (-pair[1], pair[0]))
    return pairs[:n]

In [ ]:
text = "The cat sat. The cat ran! A dog sat."
check(top_words(text, 2, set()), [("cat", 2), ("sat", 2)])
check(top_words(text, 3, set()), [("cat", 2), ("sat", 2), ("the", 2)])
check(top_words(text, 10, {"the", "a"}), [("cat", 2), ("sat", 2), ("dog", 1), ("ran", 1)])
check(top_words("Tea, tea, TEA and coffee", 1, {"and"}), [("tea", 3)])
check(top_words("b a c b a b", 3, set()), [("b", 3), ("a", 2), ("c", 1)])
check(top_words("", 5, set()), [])
check(top_words("the the the", 2, {"the"}), [])

## Project 2, step 3: ⭐ Stretch: the full report

Reuse `words` for both counting words and deciding whether a piece is a real sentence. Turn all three end marks into one, so a single `split` does the cutting. `max` with `key=len` already keeps the first of equally long words.

In [ ]:
def count_sentences(text):
    text = text.replace("!", ".").replace("?", ".")
    count = 0
    for piece in text.split("."):
        if len(words(piece)) > 0:
            count += 1
    return count


def report(text):
    all_words = words(text)
    sentences = count_sentences(text)

    if sentences == 0:
        average = 0.0
    else:
        average = round(len(all_words) / sentences, 1)

    if len(all_words) == 0:
        longest = ""
    else:
        longest = max(all_words, key=len)

    if average < 8:
        level = "easy"
    elif average <= 15:
        level = "medium"
    else:
        level = "hard"

    return {"words": len(all_words), "sentences": sentences, "average": average,
            "longest_word": longest, "level": level}

In [ ]:
check(report("I like tea. Do you like tea too?"),
      {"words": 8, "sentences": 2, "average": 4.0, "longest_word": "like", "level": "easy"})
check(report("Wait... what?!"),
      {"words": 2, "sentences": 2, "average": 1.0, "longest_word": "wait", "level": "easy"})
check(report("hello there"),
      {"words": 2, "sentences": 1, "average": 2.0, "longest_word": "hello", "level": "easy"})
check(report("One two three four five six seven eight. Nine!"),
      {"words": 9, "sentences": 2, "average": 4.5, "longest_word": "three", "level": "easy"})
check(report("a b c d e f g h i j k l m n o p q"),
      {"words": 17, "sentences": 1, "average": 17.0, "longest_word": "a", "level": "hard"})
check(report("a b c d e f g h. i j k l m n o p q r s."),
      {"words": 19, "sentences": 2, "average": 9.5, "longest_word": "a", "level": "medium"})
check(report(""),
      {"words": 0, "sentences": 0, "average": 0.0, "longest_word": "", "level": "easy"})

---

# Project 3: a tic-tac-toe engine

## Project 3, step 1: a board that checks moves

Guard first, act last: every illegal case returns `False` before the grid is touched. The range check must reject `-1` on purpose, because Python would happily accept it as "the last row".

In [ ]:
class Board:
    def __init__(self):
        self.grid = [["."] * 3 for _ in range(3)]

    def place(self, row, col, mark):
        if not (0 <= row <= 2 and 0 <= col <= 2):
            return False
        if mark != "X" and mark != "O":
            return False
        if self.grid[row][col] != ".":
            return False
        self.grid[row][col] = mark
        return True

    def show(self):
        rows = []
        for r in self.grid:
            rows.append("".join(r))
        return "\n".join(rows)

In [ ]:
b = Board()
check(b.show(), "...\n...\n...")
check(b.place(0, 0, "X"), True)
check(b.place(1, 2, "O"), True)
check(b.show(), "X..\n..O\n...")
check(b.place(0, 0, "O"), False)     # taken
check(b.place(3, 0, "X"), False)     # off the board
check(b.place(-1, 0, "X"), False)    # off the board too
check(b.place(2, 2, "Z"), False)     # not a real mark
check(b.show(), "X..\n..O\n...")   # nothing changed
check(Board().show(), "...\n...\n...")   # a new board starts empty

## Project 3, step 2: find the winner

Don't write 8 separate `if`s. Gather the 8 lines into one list, then one loop checks them all the same way. The `!= "."` test stops three empty cells from "winning".

In [ ]:
def all_lines(grid):
    lines = []
    for i in range(3):
        lines.append(grid[i])                                  # row i
        lines.append([grid[0][i], grid[1][i], grid[2][i]])     # column i
    lines.append([grid[0][0], grid[1][1], grid[2][2]])
    lines.append([grid[0][2], grid[1][1], grid[2][0]])
    return lines


def winner(grid):
    for line in all_lines(grid):
        if line[0] != "." and line[0] == line[1] == line[2]:
            return line[0]
    return None

In [ ]:
check(winner([["X", "X", "X"], ["O", "O", "."], [".", ".", "."]]), "X")
check(winner([["O", "X", "."], ["O", "X", "."], [".", "X", "O"]]), "X")
check(winner([["O", "X", "X"], [".", "O", "X"], [".", ".", "O"]]), "O")    # diagonal
check(winner([["X", "X", "O"], [".", "O", "."], ["O", ".", "X"]]), "O")    # other diagonal
check(winner([["X", "O", "X"], ["X", "O", "O"], ["O", "X", "X"]]), None)   # full, no winner
check(winner([[".", ".", "."], [".", ".", "."], [".", ".", "."]]), None)
check(winner([["O", ".", "."], ["O", ".", "."], ["O", ".", "."]]), "O")

## Project 3, step 3: a game that takes turns

A class that remembers: board, current player and "over" all live on `self`. `play` checks things in a fixed order, and switches the turn only after a normal move, never after a refused one.

In [ ]:
class Game:
    def __init__(self):
        self.board = Board()
        self.turn = "X"
        self.over = False

    def board_full(self):
        for row in self.board.grid:
            if "." in row:
                return False
        return True

    def play(self, row, col):
        if self.over:
            return "Game over"
        if not self.board.place(row, col, self.turn):
            return "Not allowed"
        if winner(self.board.grid) == self.turn:
            self.over = True
            return self.turn + " wins!"
        if self.board_full():
            self.over = True
            return "Draw!"
        player = self.turn
        if self.turn == "X":
            self.turn = "O"
        else:
            self.turn = "X"
        return player + " placed at " + str(row) + "," + str(col)

In [ ]:
def play_all(moves):
    # plays each (row, col) on a new game and collects what play() says
    game = Game()
    messages = []
    for row, col in moves:
        messages.append(game.play(row, col))
    return messages

check(play_all([(0, 0), (0, 0), (1, 1)]),
      ["X placed at 0,0", "Not allowed", "O placed at 1,1"])
check(play_all([(0, 0), (1, 0), (0, 1), (1, 1), (0, 2)]),
      ["X placed at 0,0", "O placed at 1,0", "X placed at 0,1", "O placed at 1,1", "X wins!"])
check(play_all([(0, 0), (1, 0), (0, 1), (1, 1), (2, 2), (1, 2), (2, 0)]),
      ["X placed at 0,0", "O placed at 1,0", "X placed at 0,1", "O placed at 1,1",
       "X placed at 2,2", "O wins!", "Game over"])
check(play_all([(0, 0), (0, 1), (0, 2), (1, 1), (1, 0), (1, 2), (2, 1), (2, 0), (2, 2)])[-1], "Draw!")
check(play_all([(0, 0), (0, 1), (0, 2), (1, 1), (1, 0), (1, 2), (2, 1), (2, 0), (2, 2), (0, 0)])[-1], "Game over")
check(play_all([(5, 5), (1, 1)]), ["Not allowed", "X placed at 1,1"])

## Project 3, step 4: ⭐ Stretch: a computer player

"Try a move, ask the referee, undo the move" answers both "can I win?" and "can they win?", so one helper does both jobs. Undoing straight after each try keeps the caller's grid unchanged.

In [ ]:
def winning_cell(grid, player):
    for r in range(3):
        for c in range(3):
            if grid[r][c] == ".":
                grid[r][c] = player          # try it
                won = winner(grid) == player
                grid[r][c] = "."             # undo it
                if won:
                    return (r, c)
    return None


def best_move(grid, mark):
    if mark == "X":
        other = "O"
    else:
        other = "X"

    move = winning_cell(grid, mark)
    if move is not None:
        return move
    move = winning_cell(grid, other)
    if move is not None:
        return move
    if grid[1][1] == ".":
        return (1, 1)
    for r in range(3):
        for c in range(3):
            if grid[r][c] == ".":
                return (r, c)
    return None

In [ ]:
g = [["X", "X", "."], ["O", "O", "."], [".", ".", "."]]
check(best_move(g, "X"), (0, 2))                 # win
check(best_move(g, "O"), (1, 2))                 # O wins too, on its own turn
check(g, [["X", "X", "."], ["O", "O", "."], [".", ".", "."]])   # grid not changed

check(best_move([["X", ".", "."], [".", "O", "."], [".", ".", "X"]], "O"), (0, 1))   # nothing to win or block
check(best_move([["X", ".", "."], [".", ".", "."], ["X", ".", "."]], "O"), (1, 0))   # block
check(best_move([["X", ".", "."], [".", ".", "."], [".", ".", "."]], "O"), (1, 1))   # centre
check(best_move([["X", "O", "X"], ["X", "O", "O"], ["O", "X", "X"]], "X"), None)     # full

---

# Project 4: the expense diary

## Project 4, step 1: read the diary safely

Validate one rule at a time and skip the line as soon as one fails. The date check is fiddly enough to deserve its own helper, and `.isdigit()` rejects negatives and decimals for free.

In [ ]:
def good_date(text):
    parts = text.split("-")
    if len(parts) != 3:
        return False
    year, month, day = parts
    if not (year.isdigit() and month.isdigit() and day.isdigit()):
        return False
    return len(year) == 4 and len(month) == 2 and len(day) == 2


def parse_expenses(lines):
    expenses = []
    for line in lines:
        parts = line.split(",")
        if len(parts) != 3:
            continue
        date = parts[0].strip()
        category = parts[1].strip().lower()
        amount = parts[2].strip()
        if not good_date(date) or category == "" or not amount.isdigit():
            continue
        expenses.append((date, category, int(amount)))
    return expenses

In [ ]:
check(parse_expenses(["2026-10-01,food,250", "2026-10-01, Travel ,80", "2026-10-03,food,abc"]),
      [("2026-10-01", "food", 250), ("2026-10-01", "travel", 80)])
check(parse_expenses(["2026-10-05,FOOD,  1200 "]), [("2026-10-05", "food", 1200)])
check(parse_expenses(["2026-10-05,food,-50", "2026-10-05,food,12.5"]), [])
check(parse_expenses(["2026-10-05,food", "2026-10-05,food,10,extra", ""]), [])
check(parse_expenses(["05-10-2026,food,10", "2026-1-05,food,10", "2026-10-xx,food,10"]), [])
check(parse_expenses(["2026-10-05, ,10"]), [])
check(parse_expenses([]), [])

## Project 4, step 2: totals by category and month

One pass, two running totals. `defaultdict(int)` saves the "is this key new?" check. Slicing the date to 7 characters keeps the year, so December 2025 and December 2026 stay apart.

In [ ]:
from collections import defaultdict


def totals(expenses):
    by_category = defaultdict(int)
    by_month = defaultdict(int)
    for date, category, amount in expenses:
        by_category[category] += amount
        by_month[date[:7]] += amount
    return (dict(by_category), dict(by_month))

In [ ]:
check(totals([("2026-10-01", "food", 250), ("2026-10-02", "travel", 80), ("2026-11-01", "food", 100)]),
      ({"food": 350, "travel": 80}, {"2026-10": 330, "2026-11": 100}))
check(totals([("2026-12-31", "gifts", 500)]), ({"gifts": 500}, {"2026-12": 500}))
check(totals([("2025-12-31", "food", 10), ("2026-12-01", "food", 20)]),
      ({"food": 30}, {"2025-12": 10, "2026-12": 20}))    # same month number, different years
check(totals([]), ({}, {}))

## Project 4, step 3: ⭐ Stretch: the monthly report

Group by month, then treat each month as its own small report. `totals` already gives the per-category spending for one month. The two "biggest first, ties by name/date" orders both use the `(-number, text)` sort key from project 2.

In [ ]:
def biggest_day(expenses):
    per_day = defaultdict(int)
    for date, category, amount in expenses:
        per_day[date] += amount
    ranked = sorted(per_day.items(), key=lambda p: (-p[1], p[0]))
    return ranked[0]          # (day, total)


def monthly_report(lines, budgets):
    by_month = defaultdict(list)
    for expense in parse_expenses(lines):
        by_month[expense[0][:7]].append(expense)

    report = []
    for month in sorted(by_month):
        expenses = by_month[month]
        by_category, month_total = totals(expenses)
        day, day_total = biggest_day(expenses)
        report.append(month + ": spent " + str(month_total[month])
                      + ", biggest day " + day + " (" + str(day_total) + ")")

        overs = []
        for category, spent in by_category.items():
            if category in budgets and spent > budgets[category]:
                overs.append((spent - budgets[category], category, spent))
        overs.sort(key=lambda o: (-o[0], o[1]))
        for over, category, spent in overs:
            report.append("  ! " + category + " over by " + str(over)
                          + " (" + str(spent) + " of " + str(budgets[category]) + ")")
    return report

In [ ]:
lines = ["2026-10-01,food,300", "2026-10-05,food,900", "2026-10-05,travel,0",
         "2026-10-09,fun,330", "2026-11-02,food,50"]
budgets = {"food": 1000, "fun": 300}
check(monthly_report(lines, budgets),
      ["2026-10: spent 1530, biggest day 2026-10-05 (900)",
       "  ! food over by 200 (1200 of 1000)",
       "  ! fun over by 30 (330 of 300)",
       "2026-11: spent 50, biggest day 2026-11-02 (50)"])

# a tie for the biggest day goes to the earliest; exactly on budget is fine; bad lines skipped
check(monthly_report(["2026-12-03,food,100", "2026-12-01,fun,60", "2026-12-01,fun,40", "oops"],
                     {"food": 100}),
      ["2026-12: spent 200, biggest day 2026-12-01 (100)"])

# months in date order, even if the diary isn't; tied days go to the earliest; equal overspends go alphabetically
check(monthly_report(["2027-01-01,b,150", "2026-12-01,a,10", "2027-01-02,a,150"],
                     {"a": 100, "b": 100}),
      ["2026-12: spent 10, biggest day 2026-12-01 (10)",
       "2027-01: spent 300, biggest day 2027-01-01 (150)",
       "  ! a over by 50 (150 of 100)",
       "  ! b over by 50 (150 of 100)"])

check(monthly_report([], {"food": 100}), [])